# Animation Interpolation

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Dataset

In [2]:
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as transforms
import random
from pathlib import Path
from typing import Any, Dict, List, Literal, Tuple, Optional, TypeAlias

DatasetSplit: TypeAlias = Literal['train', 'val', 'test']
TripletData: TypeAlias = Dict[str, Any]

class ATDFrameInterpolationDataset(Dataset):
    def __init__(
        self,
        root_dir: str,
        split: DatasetSplit = 'train',
        transform: transforms.Compose | None = None,
        image_size: Tuple[int, int] = (256, 256),
        max_triplets: int | None = None
    ) -> None:
        self.root_dir = Path(root_dir)
        self.split = split
        self.image_size = image_size
        self.max_triplets = max_triplets

        if transform is None:
            self.transform = transforms.Compose([
                transforms.Resize(image_size),
                transforms.ToTensor(),
            ])
        else:
            self.transform = transform

        self.triplet_folders = self._find_triplet_folders()
        self._create_splits()

        print(f"Created {split} dataset with {len(self.data)} triplets")
        print(f"Found {len(self.triplet_folders)} total triplet folders")

    def _find_triplet_folders(self) -> List[Path]:
        triplet_folders = []

        for item in self.root_dir.iterdir():
            if item.is_dir():
                im1_path = item / "frame1.jpg"
                im2_path = item / "frame2.jpg"
                im3_path = item / "frame3.jpg"

                if im1_path.exists() and im2_path.exists() and im3_path.exists():
                    triplet_folders.append(item)

        if len(triplet_folders) == 0:
            print(f"Warning: No triplet folders found in {self.root_dir}")

        return sorted(triplet_folders)

    def _create_splits(self):
        random.seed(42)
        shuffled_folders = self.triplet_folders.copy()
        random.shuffle(shuffled_folders)

        if self.max_triplets:
            shuffled_folders = shuffled_folders[:self.max_triplets]

        total = len(shuffled_folders)
        train_end = int(0.7 * total)
        val_end = int(0.85 * total)

        if self.split == 'train':
            self.data = shuffled_folders[:train_end]
        elif self.split == 'val':
            self.data = shuffled_folders[train_end:val_end]
        elif self.split == 'test':
            self.data = shuffled_folders[val_end:]
        else:
            raise ValueError(f"Unknown split: {self.split}")

    def _get_frame_paths(self, folder_path: Path) -> Tuple[Path, Path, Path]:
        im1_path = folder_path / "frame1.jpg"
        im2_path = folder_path / "frame2.jpg"
        im3_path = folder_path / "frame3.jpg"

        if im1_path.exists() and im2_path.exists() and im3_path.exists():
            return im1_path, im2_path, im3_path

        raise ValueError(f"Could not find 3 frames in {folder_path}")

    def __len__(self) -> int:
        return len(self.data)

    def __getitem__(self, idx: int) -> TripletData:
        """
        Returns:
            dict with keys:
                'frame1': First frame [C, H, W] - Input
                'frame2': Middle frame [C, H, W] - Target (ground truth)
                'frame3': Third frame [C, H, W] - Input
                'triplet_name': Name of the triplet folder
                'frame_paths': List of paths to the frames
        """
        folder_path = self.data[idx]
        frame1_path, frame2_path, frame3_path = self._get_frame_paths(folder_path)

        frame1 = Image.open(frame1_path).convert('RGB')
        frame2 = Image.open(frame2_path).convert('RGB')
        frame3 = Image.open(frame3_path).convert('RGB')

        frame1 = self.transform(frame1)
        frame2 = self.transform(frame2)
        frame3 = self.transform(frame3)

        return {
            'frame1': frame1,
            'frame2': frame2,
            'frame3': frame3,
            'triplet_name': folder_path.name,
            'frame_paths': [str(frame1_path), str(frame2_path), str(frame3_path)]
        }

In [3]:
def create_dataloaders(
    dataset_root: str,
    batch_size: int = 8,
    image_size: Tuple[int, int] = (256, 256),
    num_workers: int = 4,
    max_triplets: Optional[int] = None
) -> Tuple[DataLoader, DataLoader, DataLoader]:

    transform = transforms.Compose([
        transforms.Resize(image_size),
        transforms.ToTensor(),
    ])

    train_dataset = ATDFrameInterpolationDataset(
        root_dir=dataset_root,
        split='train',
        transform=transform,
        image_size=image_size,
        max_triplets=max_triplets
    )

    val_dataset = ATDFrameInterpolationDataset(
        root_dir=dataset_root,
        split='val',
        transform=transform,
        image_size=image_size,
        max_triplets=max_triplets
    )

    test_dataset = ATDFrameInterpolationDataset(
        root_dir=dataset_root,
        split='test',
        transform=transform,
        image_size=image_size,
        max_triplets=max_triplets
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
        drop_last=True,
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
    )

    return train_loader, val_loader, test_loader

### Model Architecture

In [4]:
import torch
import torch.nn as nn


class ConvBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int):
        super(ConvBlock, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.conv(x)


class Down(nn.Module):
    def __init__(self, in_channels: int, out_channels: int):
        super(Down, self).__init__()
        self.conv = ConvBlock(in_channels, out_channels)
        self.pool = nn.MaxPool2d(kernel_size=2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        features = self.conv(x)
        pooled = self.pool(features)
        return features, pooled


class Up(nn.Module):
    def __init__(self, in_channels: int, out_channels: int):
        super(Up, self).__init__()
        self.upconv = nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)
        self.conv = ConvBlock(in_channels, out_channels)

    def forward(self, x: torch.Tensor, skip: torch.Tensor) -> torch.Tensor:
        x = self.upconv(x)  # Upsample
        x = torch.cat((x, skip), dim=1)  # Concatenate with skip connection
        return self.conv(x)

In [5]:
from torchvision.models.optical_flow import raft_large, Raft_Large_Weights

class FrameInterpolationNet(nn.Module):
    def __init__(self, use_optical_flow=True, freeze_flow=True):
        super(FrameInterpolationNet, self).__init__()

        self.use_optical_flow = use_optical_flow

        if use_optical_flow:
            self.flow_net = raft_large(weights=Raft_Large_Weights.DEFAULT)
            if freeze_flow:
                for param in self.flow_net.parameters():
                    param.requires_grad = False
            self.flow_net.eval()

        input_channels = 6
        if use_optical_flow:
            input_channels += 2

        self.input_conv = nn.Conv2d(input_channels, 64, kernel_size=7, padding=3, stride=1)
        self.input_bn = nn.BatchNorm2d(64)
        self.input_relu = nn.ReLU(inplace=True)

        # Encoder
        self.down1 = Down(64, 64)
        self.down2 = Down(64, 128)
        self.down3 = Down(128, 256)
        self.down4 = Down(256, 512)

        # Bottleneck
        self.bottleneck = ConvBlock(512, 1024)

        # Decoder
        self.up1 = Up(1024, 512)
        self.up2 = Up(512, 256)
        self.up3 = Up(256, 128)
        self.up4 = Up(128, 64)

        # Final output layer
        self.final_conv = nn.Conv2d(64, 3, kernel_size=1)


    def forward(self, frame1: torch.Tensor, frame2: torch.Tensor) -> torch.Tensor:
        """
        Args:
            frame1: [B, 3, H, W] - first frame
            frame2: [B, 3, H, W] - second frame
        Returns:
            interpolated_frame: [B, 3, H, W] - interpolated frame
        """

        x = torch.cat([frame1, frame2], dim=1)  # [B, 6, H, W]

        if self.use_optical_flow:
            with torch.no_grad():
                frame1_255 = (frame1 * 255).clamp(0, 255)
                frame2_255 = (frame2 * 255).clamp(0, 255)

                flow_12 = self.flow_net(frame1_255, frame2_255)[-1]  # [B, 2, H, W]

            x = torch.cat([x, flow_12], dim=1)  # [B, 8, H, W]

        x = self.input_conv(x)
        x = self.input_bn(x)
        x = self.input_relu(x)

        # Encoder (Downsampling)
        skip1, d1 = self.down1(x)
        skip2, d2 = self.down2(d1)
        skip3, d3 = self.down3(d2)
        skip4, d4 = self.down4(d3)

        # Bottleneck
        bottleneck = self.bottleneck(d4)

        # Decoder (Upsampling with Skip Connections)
        up1 = self.up1(bottleneck, skip4)
        up2 = self.up2(up1, skip3)
        up3 = self.up3(up2, skip2)
        up4 = self.up4(up3, skip1)

        # Final output
        output = self.final_conv(up4) # [B, 3, H, W]
        output = torch.sigmoid(output)

        return output


In [6]:
import torch.nn.functional as F
from torchvision.models import vgg19, VGG19_Weights

class InterpolationLoss(nn.Module):
    def __init__(self, l1_weight: float = 1.0, perceptual_weight: float = 0.1):
        super().__init__()
        self.l1_weight = l1_weight
        self.perceptual_weight = perceptual_weight

        vgg: nn.Module = vgg19(weights=VGG19_Weights.DEFAULT).features[:16]
        self.vgg = vgg.eval()
        for param in self.vgg.parameters():
            param.requires_grad = False

    def forward(self, pred: torch.Tensor, target: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        l1_loss = F.l1_loss(pred, target)

        pred_features = self.vgg(pred)
        target_features = self.vgg(target)
        perceptual_loss = F.mse_loss(pred_features, target_features)

        total_loss = (
            self.l1_weight * l1_loss + self.perceptual_weight * perceptual_loss
        )

        return total_loss, l1_loss, perceptual_loss

### Evaluation and Plotting

In [7]:
import numpy as np
import torch

def calculate_psnr(pred: torch.Tensor, target: torch.Tensor) -> float:
    mse = torch.mean((pred - target) ** 2)
    if mse == 0:
        return float('inf')
    psnr = 20 * torch.log10(1.0 / torch.sqrt(mse))
    return psnr.item()


def calculate_ssim(pred: torch.Tensor, target: torch.Tensor) -> float:
    pred_gray = torch.mean(pred, dim=0, keepdim=True)
    target_gray = torch.mean(target, dim=0, keepdim=True)

    mu1 = torch.mean(pred_gray)
    mu2 = torch.mean(target_gray)

    sigma1_sq = torch.var(pred_gray)
    sigma2_sq = torch.var(target_gray)
    sigma12 = torch.mean((pred_gray - mu1) * (target_gray - mu2))

    c1 = 0.01 ** 2
    c2 = 0.03 ** 2

    numerator = (2 * mu1 * mu2 + c1) * (2 * sigma12 + c2)
    denominator = (mu1**2 + mu2**2 + c1) * (sigma1_sq + sigma2_sq + c2)
    ssim = numerator / denominator

    return ssim.item()


def calculate_batch_psnr(pred_batch: torch.Tensor, target_batch: torch.Tensor) -> float:
    psnr_values = [calculate_psnr(pred_batch[i], target_batch[i])
                   for i in range(pred_batch.shape[0])]
    return np.mean(psnr_values)


def calculate_batch_ssim(pred_batch: torch.Tensor, target_batch: torch.Tensor) -> float:
    ssim_values = [calculate_ssim(pred_batch[i], target_batch[i])
                   for i in range(pred_batch.shape[0])]
    return np.mean(ssim_values)

In [8]:
import os
import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image

def tensor_to_image(tensor: torch.Tensor) -> np.ndarray:
    if len(tensor.shape) == 4:
        tensor = tensor[0]

    tensor = tensor.detach().cpu()
    image = tensor.permute(1, 2, 0).numpy()
    image = np.clip(image, 0, 1)

    return image

def plot_prediction_batch(
    model: nn.Module,
    batch: torch.Tensor,
    device: str = 'cuda',
    max_samples: int = 4,
    figsize: Tuple[int, int] =(20, 12),
    save_path: str | None = None,
    show_metrics: bool = True,
) -> plt.Figure:
    model.eval()

    with torch.no_grad():
        frame1 = batch['frame1'].to(device)
        frame2 = batch['frame2'].to(device)
        frame3 = batch['frame3'].to(device)

        predicted = model(frame1, frame3)

        batch_size = min(frame1.shape[0], max_samples)

        # 5 columns (frame1, predicted, GT, frame3, error)
        fig, axes = plt.subplots(batch_size, 5, figsize=figsize)

        if batch_size == 1:
            axes = axes.reshape(1, -1)

        for i in range(batch_size):
            img_frame1 = tensor_to_image(frame1[i])
            img_predicted = tensor_to_image(predicted[i])
            img_gt = tensor_to_image(frame2[i])
            img_frame3 = tensor_to_image(frame3[i])

            error_map = np.abs(img_predicted - img_gt)
            error_map = np.mean(error_map, axis=2)

            axes[i, 0].imshow(img_frame1)
            axes[i, 0].set_title('Frame 1 (Input)', fontsize=10)
            axes[i, 0].axis('off')

            axes[i, 1].imshow(img_predicted)
            title = 'Predicted'
            if show_metrics:
                psnr_val = calculate_psnr(predicted[i], frame2[i])
                ssim_val = calculate_ssim(predicted[i], frame2[i])
                title += f'\nPSNR: {psnr_val:.1f}dB\nSSIM: {ssim_val:.3f}'
            axes[i, 1].set_title(title, fontsize=10)
            axes[i, 1].axis('off')

            axes[i, 2].imshow(img_gt)
            axes[i, 2].set_title('Ground Truth', fontsize=10)
            axes[i, 2].axis('off')

            axes[i, 3].imshow(img_frame3)
            axes[i, 3].set_title('Frame 3 (Input)', fontsize=10)
            axes[i, 3].axis('off')

            im = axes[i, 4].imshow(error_map, cmap='hot', vmin=0, vmax=0.5)
            axes[i, 4].set_title('Error Map', fontsize=10)
            axes[i, 4].axis('off')

            if i == 0:
                plt.colorbar(im, ax=axes[i, 4], shrink=0.8)

        if show_metrics:
            batch_psnr = calculate_batch_psnr(predicted, frame2)
            batch_ssim = calculate_batch_ssim(predicted, frame2)
            fig.suptitle(f'Frame Interpolation Results\n'
                        f'Batch Average - PSNR: {batch_psnr:.2f}dB, SSIM: {batch_ssim:.3f}',
                        fontsize=14, y=0.98)
        else:
            fig.suptitle('Frame Interpolation Results', fontsize=14, y=0.98)

        plt.tight_layout()

        if save_path:
            plt.savefig(save_path, dpi=150, bbox_inches='tight')
            print(f"Saved visualization to {save_path}")

        plt.show()

    return fig

def create_gif_from_triplet(
    frame1: torch.Tensor,
    predicted: torch.Tensor,
    frame3,
    save_path,
    duration=500
):
    img1 = tensor_to_image(frame1)
    img_pred = tensor_to_image(predicted)
    img3 = tensor_to_image(frame3)

    img1_pil = Image.fromarray((img1 * 255).astype(np.uint8))
    img_pred_pil = Image.fromarray((img_pred * 255).astype(np.uint8))
    img3_pil = Image.fromarray((img3 * 255).astype(np.uint8))

    frames = [img1_pil, img_pred_pil, img3_pil]
    frames[0].save(save_path, save_all=True, append_images=frames[1:],
                    duration=duration, loop=0)

    print(f"Saved animation to {save_path}")


def evaluate_model_metrics(
    model: nn.Module,
    dataloader: DataLoader,
    max_batches: int | None = None,
    device: str = "cuda",
) -> Tuple[float, float]:
    model.eval()

    psnr_values = []
    ssim_values = []

    with torch.no_grad():
        for i, batch in enumerate(dataloader):
            if max_batches and i >= max_batches:
                break

            frame1 = batch['frame1'].to(device)
            frame2 = batch['frame2'].to(device)
            frame3 = batch['frame3'].to(device)

            predicted = model(frame1, frame3)

            batch_psnr = calculate_batch_psnr(predicted, frame2)
            batch_ssim = calculate_batch_ssim(predicted, frame2)

            psnr_values.append(batch_psnr)
            ssim_values.append(batch_ssim)

    avg_psnr = np.mean(psnr_values)
    avg_ssim = np.mean(ssim_values)

    print(f"Dataset Evaluation Results:")
    print(f"Average PSNR: {avg_psnr:.2f} dB")
    print(f"Average SSIM: {avg_ssim:.3f}")

    return avg_psnr, avg_ssim

In [9]:
def visual_eval(
    model: nn.Module,
    dataloader: DataLoader,
    gifs: int = 3,
    device: str = "cuda",
    save_dir: str | None = None
):
    if save_dir:
        os.makedirs(save_dir, exist_ok=True)

    batch = next(iter(dataloader))

    save_path = os.path.join(save_dir, "predictions.png") if save_dir else None
    plot_prediction_batch(
        model,
        batch,
        device=device,
        max_samples=4,
        save_path=save_path
    )

    gifs_count = min(gifs, batch['frame1'].shape[0])

    if save_dir:
        model.eval()
        with torch.no_grad():
            frame1 = batch['frame1'][:gifs_count].to(device)
            frame3 = batch['frame3'][:gifs_count].to(device)
            predicted = model(frame1, frame3)

            for i in range(gifs_count):
                gif_path = os.path.join(save_dir, f'interpolation_{i}.gif')
                create_gif_from_triplet(frame1[i], predicted[i], frame3[i], gif_path)

### Training

In [10]:
!pip install -q neptune

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 4.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 487.9/487.9 kB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.9/139.9 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 121.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.2/85.2 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 6.1 MB/s eta 0:00:00


In [11]:
NEPTUNE_PROJECT_NAME = "sigk/animation-interpolation"
NEPTUNE_API_KEY = input("Enter Neptune API key: ")

Enter Neptune API key: eyJhcGlfYWRkcmVzcyI6Imh0dHBzOi8vYXBwLm5lcHR1bmUuYWkiLCJhcGlfdXJsIjoiaHR0cHM6Ly9hcHAubmVwdHVuZS5haSIsImFwaV9rZXkiOiI4ODA4YThkMy04N2Y5LTQ4NzItYmExMi1hNDdkM2Y4ZTJiNWYifQ==


In [12]:
import neptune

def train_eval_loop(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    loss_fn: nn.Module,
    epochs: int = 10,
    run: neptune.Run | None = None,
    device: str = "cuda",
):

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0

        for batch in train_loader:
            frame1 = batch['frame1'].to(device)
            frame2 = batch['frame2'].to(device)
            frame3 = batch['frame3'].to(device)

            optimizer.zero_grad()
            predicted = model(frame1, frame3)

            loss, _, _ = loss_fn(predicted, frame2)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

            if run:
                run["train/step/loss"].log(total_loss / len(train_loader))

        avg_train_loss = total_loss / len(train_loader)

        if run:
            run["train/epoch/loss"].log(avg_train_loss)

        model.eval()
        with torch.no_grad():
            val_psnr, val_ssim = evaluate_model_metrics(model, val_loader, device=device)
            predictions = plot_prediction_batch(model, next(iter(val_loader)), device=device, show_metrics=False)

            if run:
                run["val/epoch/psnr"].log(val_psnr)
                run["val/epoch/ssim"].log(val_ssim)
                run["val/epoch/predictions"].upload(predictions)

        print(f"Epoch [{epoch+1}/{epochs}] | Validation PSNR: {val_psnr:.2f}, SSIM: {val_ssim:.3f} | Train Loss: {avg_train_loss:.4f}")

In [13]:
# DATASET_ROOT = "C:\\Users\\kacpe\\Code\\Uni\\25L\\SIGK\\data\\atd_12k\\datasets\\train_10k"
DATASET_ROOT = "/content/drive/MyDrive/data/atd_12k/datasets/train_10k"

In [19]:
batch_size = 8
image_size = (128, 128)

train_loader, val_loader, test_loader = create_dataloaders(
    dataset_root=DATASET_ROOT,
    batch_size=batch_size,
    image_size=image_size,
    num_workers=2,
)

print(f"Train loader: {len(train_loader)} batches")
print(f"Val loader: {len(val_loader)} batches")
print(f"Test loader: {len(test_loader)} batches")

Created train dataset with 7000 triplets
Found 10000 total triplet folders
Created val dataset with 1500 triplets
Found 10000 total triplet folders
Created test dataset with 1500 triplets
Found 10000 total triplet folders
Train loader: 875 batches
Val loader: 188 batches
Test loader: 188 batches


In [20]:
from torch.optim import AdamW

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_flow=False
model = FrameInterpolationNet(use_optical_flow=use_flow).to(device)
lr=1e-4
weight_decay=1e-4
optimizer = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

total_params = sum(p.numel() for p in model.parameters())
print(f"Total model parameters: {total_params / 1e6:.2f}M")

Total model parameters: 31.10M


In [21]:
epochs = 15
criterion = InterpolationLoss().to(device)

run = neptune.init_run(
    project=NEPTUNE_PROJECT_NAME,
    api_token=NEPTUNE_API_KEY,
)

run["model"] = model.__class__.__name__
run["parameters"] = total_params
run["use_flow"] = use_flow
run["optimizer"] = optimizer.__class__.__name__
run["lr"] = lr
run["weight_decay"] = weight_decay
run["loss_fn"] = criterion.__class__.__name__
run["epochs"] = epochs
run["batch_size"] = batch_size
run["image_size"] = str(image_size)
run["dataset_size"] = len(train_loader.dataset)

[neptune] [info   ] Neptune initialized. Open in the app: https://app.neptune.ai/sigk/animation-interpolation/e/AN-4


In [22]:
train_eval_loop(
    model,
    train_loader,
    val_loader,
    optimizer,
    criterion,
    epochs=epochs,
    run=run,
    device=device,
)

KeyboardInterrupt: 

In [ ]:
test_psnr, test_ssim = evaluate_model_metrics(model, test_loader, device=device)

In [ ]:
run["test/psnr"].log(test_psnr)
run["test/ssim"].log(test_ssim)

In [ ]:
gifs = 8
visual_eval(model, test_loader, device="cuda", gifs=8, save_dir="./results")

In [ ]:
run["test/predictions"].upload("./results/predictions.png")

In [ ]:
for gif in range(gifs):
    gif_path = os.path.join("./results", f'interpolation_{gif}.gif')
    run[f"test/gif_{gif}"].upload(gif_path)

In [ ]:
run.stop()